# IVD comparison bar plots

Draws one bar per comparison on the **Summary** sheet of `IVD_comparisons_with_genes.xlsx`, in two versions: with the
expected-by-chance tick (folder `with_expected_by_chance/`) and without it (`without_expected_by_chance/`, files ending in `_no_chance`).

- **dark red**: genes changed in the same direction, up in both lists (higher in decidual / decidualized cells);
- **red**: same direction, down in both;
- **grey**: opposite direction;
- **black tick** (first version only): % expected by chance, from the Summary sheet (the agreement expected between two unrelated lists with the same up/down balance);
- **bold number**: % same direction; **n**: genes compared.

The counts come from each comparison's gene sheet (named in the Summary's *genes listed on* column), because the Summary's
counts are formulas that the file doesn't store values for. The notebook checks every gene sheet's *same direction* column
and the Summary's chance level against its own counts, so a sheet that doesn't match the Summary stops it.

In [ ]:
from pathlib import Path
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

XLSX = Path("IVD_comparisons_with_genes.xlsx")  # on Drive: xmens_figures/2026-10-30_revision_panels/DESeq2_DE_and_IVD_comparisons/
OUT = Path("ivd_bars")                          # bar_values.csv, and a folder per version with a PNG and SVG per comparison
VERSIONS = {  # folder: (draw the expected-by-chance tick?, filename suffix)
    "with_expected_by_chance": (True, ""),
    "without_expected_by_chance": (False, "_no_chance"),
}
COLORS = dict(up="#803532", down="#b04334", opposite="#c9c9c9", chance="black", n="#6b6b6b")

for folder in VERSIONS:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
# editable text in the SVGs, and no date or random IDs in them, so a rerun writes identical files
mpl.rcParams.update({"svg.fonttype": "none", "pdf.fonttype": 42, "svg.hashsalt": "ivd-bars"})


def save(fig, stem):
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".svg"), dpi=300, bbox_inches="tight", metadata={"Date": None})

## Counts per comparison

In [ ]:
summary = pd.read_excel(XLSX, sheet_name="Summary")
rows = []
for _, s in summary.iterrows():
    sheet = s["genes listed on"].removeprefix("gene sheet: ")
    genes = pd.read_excel(XLSX, sheet_name=sheet)
    # the two direction columns are the ones that hold only "up" and "down"
    dirs = [c for c in genes.columns if genes[c].notna().any() and genes[c].dropna().isin(["up", "down"]).all()]
    assert len(dirs) == 2, (sheet, dirs)
    a, b = genes[dirs[0]], genes[dirs[1]]
    same = a == b
    assert (same == genes["same direction"].astype(bool)).all(), f"{sheet}: 'same direction' column disagrees"
    chance = 100 * ((a == "up").mean() * (b == "up").mean() + (a == "down").mean() * (b == "down").mean())
    assert round(chance, 1) == s["% expected by chance"], (sheet, chance, s["% expected by chance"])
    for col, value in (("genes compared", len(genes)), ("same direction", same.sum())):
        assert pd.isna(s[col]) or s[col] == value, (sheet, col)  # only if the file stores formula results
    first, second = re.sub(r"\s*\(.*\)$", "", s["comparison"]).split(" vs ", 1)
    rows.append(dict(comparison=s["comparison"], sheet=sheet, label=f"{first}\nvs {second}", n=len(genes),
                     same_up=int((same & (a == "up")).sum()), same_down=int((same & (a == "down")).sum()),
                     opposite=int((~same).sum()), pct_same=100 * same.mean(), pct_chance=s["% expected by chance"]))
bars = pd.DataFrame(rows)
bars.to_csv(OUT / "bar_values.csv", index=False)
bars.drop(columns=["comparison", "label"]).round(1)

## Drawing

In [ ]:
def draw_bar(ax, r, y=0, height=0.8, chance=True):
    up, down = 100 * r.same_up / r.n, 100 * r.same_down / r.n
    ax.barh(y, 100, height=height, color=COLORS["opposite"])
    ax.barh(y, up, height=height, color=COLORS["up"])
    ax.barh(y, down, left=up, height=height, color=COLORS["down"])
    if chance:
        ax.plot([r.pct_chance] * 2, [y - height / 2, y + height / 2], color=COLORS["chance"], lw=3, solid_capstyle="butt")
    # % label just right of the red; moved past the chance tick when the tick sits right there
    x = r.pct_chance + 1.5 if chance and 0 <= r.pct_chance - r.pct_same < 9 else r.pct_same + 2
    if x > 92:  # no room in the grey: put it inside the red
        ax.text(r.pct_same - 2, y, f"{r.pct_same:.0f}%", ha="right", va="center", fontsize=11, fontweight="bold", color="white")
    else:
        ax.text(x, y, f"{r.pct_same:.0f}%", ha="left", va="center", fontsize=11, fontweight="bold")
    ax.text(104, y, f"n={r.n:,}", ha="left", va="center", fontsize=10, color=COLORS["n"])
    ax.text(-2, y, r.label, ha="right", va="center", fontsize=10)


def finish(ax, top, bottom):
    ax.set_xlim(0, 100)
    ax.set_ylim(bottom, top)
    ax.axis("off")

## One figure per comparison

In [ ]:
for folder, (chance, suffix) in VERSIONS.items():
    for r in bars.itertuples():
        fig, ax = plt.subplots(figsize=(10, 0.65))
        draw_bar(ax, r, chance=chance)
        finish(ax, top=-0.5, bottom=0.5)
        stem = re.sub(r"[^A-Za-z0-9]+", "_", r.sheet).strip("_")
        save(fig, OUT / folder / f"{stem}{suffix}")
        plt.show()

## All comparisons in one figure

In [ ]:
legend = [mpl.patches.Patch(color=COLORS["up"], label="same direction, up in both"),
          mpl.patches.Patch(color=COLORS["down"], label="same direction, down in both"),
          mpl.patches.Patch(color=COLORS["opposite"], label="opposite direction"),
          mpl.lines.Line2D([], [], color=COLORS["chance"], lw=3, label="expected by chance")]
for folder, (chance, suffix) in VERSIONS.items():
    fig, ax = plt.subplots(figsize=(10, 0.65 * len(bars) + 0.5))
    for y, r in enumerate(bars.itertuples()):
        draw_bar(ax, r, y=y, chance=chance)
    finish(ax, top=-0.6, bottom=len(bars) - 0.4)
    handles = legend if chance else legend[:3]
    ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1), ncol=len(handles), frameon=False, fontsize=9, handlelength=1.2)
    save(fig, OUT / folder / f"all_comparisons{suffix}")
    plt.show()